# Phase A: Data Preparation for CLIP ViT-B/32 on CelebA
## Steps 1-4: Load dataset, build attribute tensor, load evaluation JSON, sanity checks


## Step 1: Load CelebA Dataset (test split only)

**Why:** We need the actual images in a standardized format with CLIP-compatible preprocessing.
- Resize to 224×224 (CLIP ViT-B/32's expected input)
- Normalize using CLIP's standard statistics
- Load only test split (~19,962 images) to evaluate on unseen data


In [ ]:
import torch
from pathlib import Path
from torchvision import datasets, transforms
import json
import os


# Determine project root (assumes notebook is in project/Phase_A_Data_Preparation.ipynb)
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent if notebook_dir.name == 'project' else notebook_dir

# CelebA dataset location: check multiple possible locations
# torchvision's CelebA expects: root/celeba/ with subdirectories (img_align_celeba, list_attr_celeba.txt, etc)
celeba_paths = [
    project_root / 'celeba',  # Option 1: In project root
    Path.home() / 'Downloads' / 'celeba',  # Option 2: In user's Downloads (already has celeba/ subdirectory)
]

data_root = None
for path in celeba_paths:
    celeba_subdir = path / 'celeba'
    # Check if the required files exist
    if (celeba_subdir / 'img_align_celeba').exists() and (celeba_subdir / 'list_attr_celeba.txt').exists():
        data_root = path
        print(f"Found CelebA at: {celeba_subdir}")
        break

if data_root is None:
    raise FileNotFoundError(f"CelebA dataset not found or corrupted in any expected location:\n" + 
                          "\n".join(str(p / 'celeba') for p in celeba_paths))

print(f"Project root: {project_root}")
print(f"CelebA data root: {data_root}")
print()

# Define transforms: resize, convert to tensor, normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # CLIP ViT-B/32 input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                        std=[0.26862954, 0.26130258, 0.27577711])
])

# Load CelebA test split
celeba = datasets.CelebA(
    root=data_root,
    split='test',
    download=False,
    transform=transform
)
print(f"  Dataset size: {len(celeba)} images")

Found CelebA at: C:\Users\alfon\Downloads\celeba\celeba
Project root: c:\Users\alfon\Desktop\Deep-Learning-Project
CelebA data root: C:\Users\alfon\Downloads\celeba

✓ CelebA test set loaded successfully
  Dataset size: 19962 images
  Expected: ~19,962 images



## Step 2: Build Attribute Tensor (test split only)

**Why:** We need structured ground truth labels for the test split only. The `list_attr_celeba.txt` file contains 40 facial attributes (-1 or 1) for each image.
- Read all attributes from file, filter to test split only
- Convert to PyTorch tensor [19962, 40]
- Index alignment: `celeba_attrs[i]` corresponds to `celeba[i]`
- Used later to validate whether retrieved images match query conditions

In [ ]:
# Get test split filenames from the dataset
test_filenames = set(celeba.filename)
print(f"Test split has {len(test_filenames)} images")

attributes = []
attr_filename_map = []
celeba_root = data_root / 'celeba'
attr_file = celeba_root / 'list_attr_celeba.txt'

print(f"Reading attributes from: {attr_file}")

with open(attr_file, 'r') as f:
    # Line 0: number of images
    num_images = int(f.readline().strip())
    print(f"  File indicates {num_images} total images in full dataset")

    # Line 1: attribute names (40 attributes)
    header_attrs = f.readline().strip().split()
    print(f"  Header attributes count: {len(header_attrs)}")
    print(f"  First 5 attributes: {header_attrs[:5]}")

    # Lines 2+: image attributes - filter to test split only
    for line in f:
        parts = line.strip().split()
        filename = parts[0]

        # Only include if in test split
        if filename in test_filenames:
            attrs = [int(x) for x in parts[1:]]  # Read as int (-1 or 1)
            attributes.append(attrs)
            attr_filename_map.append(filename)

# Convert to float32 and map -1 → 0, +1 → 1 (per CONTRACT.md section 2)
celeba_attrs = torch.tensor(attributes, dtype=torch.float32)
celeba_attrs = (celeba_attrs + 1) / 2  # Maps [-1, 1] -> [0, 1]

print(f"  Shape: {celeba_attrs.shape}")
print(f"  Expected: [{len(test_filenames)}, 40] (test split images only)")
print(f"  Data type: {celeba_attrs.dtype}")
print(f"  Value range: [{celeba_attrs.min():.1f}, {celeba_attrs.max():.1f}] (should be [0.0, 1.0])")

# Save attribute tensor for later use (relative to project root)
attr_cache_path = project_root / 'celeba_attributes_test.pt'
torch.save(celeba_attrs, attr_cache_path)

## Step 3: Load Evaluation JSON

**Why:** This JSON defines the ground truth benchmark for evaluating retrieval quality.

Each query specifies:
- **Sources:** All images matching the query condition (e.g., 4,786 smiling images for "+Smiling")
- **Valid targets:** For each source, the list of other images that are similar by attribute (Hamming distance ≤ 2) AND match the query condition

**How it's built:**
1. For each of the 4,786 smiling images (source):
   - Find all similar images using Hamming distance ≤ 2
   - Filter to only those that ALSO have "+Smiling" attribute
   - Record this list as valid targets for this source
2. Average: (targets for source1 + targets for source2 + ... ) / 4,786 = **26.0 avg targets**

**Later evaluation (Phase B):**
- Extract CLIP embeddings for all test images
- For each source, retrieve top-26 similar images using CLIP
- Check: How many of CLIP's retrievals match the ground truth targets?
- Score: Recall/Precision = (correct retrievals / expected targets)

In [3]:
eval_json_path = project_root / 'Evaluation' / 'celeba_evaluation.json'
print(f"Reading from: {eval_json_path}")

with open(eval_json_path, 'r') as f:
    queries = json.load(f)

print(f"  Total queries: {len(queries)}")

# Display summary of each query
for i, query_dict in enumerate(queries):
    query_str = query_dict['query']
    ground_truth = query_dict['ground_truth']
    num_sources = len(ground_truth)

    # Count total targets
    total_targets = sum(len(targets) for targets in ground_truth.values())
    avg_targets = total_targets / num_sources if num_sources > 0 else 0

    print(f"  Query {i}: '{query_str}'")
    print(f"    Sources: {num_sources}, Avg targets per source: {avg_targets:.1f}")

Reading from: c:\Users\alfon\Desktop\Deep-Learning-Project\Evaluation\celeba_evaluation.json
  Total queries: 14
  Query 0: '+Smiling'
    Sources: 4786, Avg targets per source: 26.0
  Query 1: '+Eyeglasses'
    Sources: 2196, Avg targets per source: 20.3
  Query 2: '-Heavy_Makeup'
    Sources: 4087, Avg targets per source: 25.2
  Query 3: '+Male'
    Sources: 1595, Avg targets per source: 64.8
  Query 4: '-Young'
    Sources: 5355, Avg targets per source: 23.3
  Query 5: '+Blond_Hair'
    Sources: 5469, Avg targets per source: 25.1
  Query 6: '+Mustache'
    Sources: 301, Avg targets per source: 7.8
  Query 7: '-Young'
    Sources: 5355, Avg targets per source: 23.3
  Query 8: '+Eyeglasses, +Smiling'
    Sources: 612, Avg targets per source: 11.6
  Query 9: '+Black_Hair, -Wavy_Hair'
    Sources: 2572, Avg targets per source: 28.7
  Query 10: '-Male, -Mustache'
    Sources: 27, Avg targets per source: 16.1
  Query 11: '+Chubby, -Young'
    Sources: 584, Avg targets per source: 9.2
  Qu

## Step 4: Sanity Check (CRITICAL)

**Why:** Index consistency is fragile—dataset indices don't match filenames.
- Verify: `celeba[13]` corresponds to filename in JSON
- Verify: attribute tensor has correct shape and values
- Verify: ground truth structure matches expectations
- **One failure here means everything downstream is wrong**


In [ ]:
# Test index from ROADMAP
test_idx = 13

print(f"Testing with index: {test_idx}")
print()

# Check 1: Dataset metadata
target_filename = celeba.filename[test_idx]
print(f"✓ Check 1: Dataset index to filename")
print(f"  celeba.filename[{test_idx}] = {target_filename}")
print()

# Check 2: Attribute tensor indexing
attr_row = celeba_attrs[test_idx]
print(f"✓ Check 2: Attribute tensor indexing")
print(f"  celeba_attrs[{test_idx}].shape = {attr_row.shape}")
print(f"  Sample attributes (first 5): {attr_row[:5].tolist()}")
print()

# Check 3: JSON ground truth structure
query_0 = queries[0]
query_str = query_0['query']
ground_truth = query_0['ground_truth']

print(f"✓ Check 3: JSON ground truth structure")
print(f"  Query 0: '{query_str}'")

targets = ground_truth[str(test_idx)]
print(f"  Source {test_idx} found in ground truth")
print(f"  Number of valid targets: {len(targets)}")
print(f"  Sample targets (first 5): {targets[:5]}")